In [ ]:
# ============================================================
# AI Based Intelligent RF Spectrum Signal Identification System
# Notebook: 05_model_training.ipynb
# Purpose: Train CNN for Automatic Modulation Classification
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    Dropout,
    Flatten,
    Dense
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

from tensorflow.keras.utils import to_categorical


# ============================================================
# 1. PROJECT PATHS
# ============================================================

processed_data_dir = "../data/processed"
models_dir = "../models"
results_dir = "../results"

os.makedirs(models_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

print("=" * 70)
print("CNN MODEL TRAINING")
print("=" * 70)

print("TensorFlow Version:", tf.__version__)


# ============================================================
# 2. LOAD PROCESSED DATA
# ============================================================

print("\nLoading processed dataset...")

X_train = np.load(
    os.path.join(processed_data_dir, "X_train.npy")
)

X_val = np.load(
    os.path.join(processed_data_dir, "X_val.npy")
)

X_test = np.load(
    os.path.join(processed_data_dir, "X_test.npy")
)

y_train = np.load(
    os.path.join(processed_data_dir, "y_train.npy")
)

y_val = np.load(
    os.path.join(processed_data_dir, "y_val.npy")
)

y_test = np.load(
    os.path.join(processed_data_dir, "y_test.npy")
)

modulation_classes = np.load(
    os.path.join(
        processed_data_dir,
        "modulation_classes.npy"
    )
)

print("Dataset loaded successfully!")

print("\nTraining data:", X_train.shape)
print("Validation data:", X_val.shape)
print("Testing data:", X_test.shape)


# ============================================================
# 3. PREPARE DATA FOR CNN
# ============================================================

num_classes = len(modulation_classes)

# Add channel dimension
# (samples, 2, 128) -> (samples, 2, 128, 1)

X_train_cnn = X_train[..., np.newaxis]
X_val_cnn = X_val[..., np.newaxis]
X_test_cnn = X_test[..., np.newaxis]

# Convert labels to categorical format

y_train_cat = to_categorical(
    y_train,
    num_classes=num_classes
)

y_val_cat = to_categorical(
    y_val,
    num_classes=num_classes
)

y_test_cat = to_categorical(
    y_test,
    num_classes=num_classes
)

print("\nCNN Input Shape:", X_train_cnn.shape[1:])
print("Number of Classes:", num_classes)


# ============================================================
# 4. BUILD CNN MODEL
# ============================================================

print("\nBuilding CNN model...")

model = Sequential([

    Input(shape=(2, 128, 1)),

    # Convolution Block 1
    Conv2D(
        filters=32,
        kernel_size=(1, 5),
        activation="relu",
        padding="same"
    ),

    BatchNormalization(),

    MaxPooling2D(
        pool_size=(1, 2)
    ),

    Dropout(0.20),


    # Convolution Block 2
    Conv2D(
        filters=64,
        kernel_size=(1, 3),
        activation="relu",
        padding="same"
    ),

    BatchNormalization(),

    MaxPooling2D(
        pool_size=(1, 2)
    ),

    Dropout(0.25),


    # Convolution Block 3
    Conv2D(
        filters=128,
        kernel_size=(1, 3),
        activation="relu",
        padding="same"
    ),

    BatchNormalization(),

    Dropout(0.30),


    # Classification Layers
    Flatten(),

    Dense(
        256,
        activation="relu"
    ),

    Dropout(0.50),

    Dense(
        128,
        activation="relu"
    ),

    Dropout(0.30),


    # Output Layer
    Dense(
        num_classes,
        activation="softmax"
    )

])


# ============================================================
# 5. COMPILE MODEL
# ============================================================

model.compile(

    optimizer="adam",

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

print("CNN model compiled successfully!")

print("\nModel Summary:")
model.summary()


# ============================================================
# 6. DEFINE CALLBACKS
# ============================================================

best_model_path = os.path.join(
    models_dir,
    "best_cnn_modulation_classifier.keras"
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

model_checkpoint = ModelCheckpoint(
    filepath=best_model_path,
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

reduce_learning_rate = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=0.000001,
    verbose=1
)


# ============================================================
# 7. TRAIN MODEL
# ============================================================

print("\n" + "=" * 70)
print("MODEL TRAINING STARTED")
print("=" * 70)

EPOCHS = 50
BATCH_SIZE = 256

history = model.fit(

    X_train_cnn,
    y_train_cat,

    validation_data=(
        X_val_cnn,
        y_val_cat
    ),

    epochs=EPOCHS,

    batch_size=BATCH_SIZE,

    callbacks=[
        early_stopping,
        model_checkpoint,
        reduce_learning_rate
    ],

    verbose=1
)


# ============================================================
# 8. SAVE FINAL TRAINED MODEL
# ============================================================

final_model_path = os.path.join(
    models_dir,
    "final_cnn_modulation_classifier.keras"
)

model.save(final_model_path)

print("\nFinal trained model saved:")
print(os.path.abspath(final_model_path))


# ============================================================
# 9. SAVE TRAINING HISTORY
# ============================================================

history_path = os.path.join(
    results_dir,
    "training_history.npz"
)

np.savez(
    history_path,

    accuracy=history.history["accuracy"],
    val_accuracy=history.history["val_accuracy"],

    loss=history.history["loss"],
    val_loss=history.history["val_loss"]
)

print("\nTraining history saved:")
print(os.path.abspath(history_path))


# ============================================================
# 10. PLOT TRAINING ACCURACY
# ============================================================

epochs_range = range(
    1,
    len(history.history["accuracy"]) + 1
)

plt.figure(figsize=(10, 6))

plt.plot(
    epochs_range,
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    epochs_range,
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.title("CNN Training and Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend()
plt.grid(True)

plt.tight_layout()

accuracy_plot_path = os.path.join(
    results_dir,
    "training_accuracy.png"
)

plt.savefig(
    accuracy_plot_path,
    dpi=300
)

plt.show()


# ============================================================
# 11. PLOT TRAINING LOSS
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    epochs_range,
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    epochs_range,
    history.history["val_loss"],
    label="Validation Loss"
)

plt.title("CNN Training and Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend()
plt.grid(True)

plt.tight_layout()

loss_plot_path = os.path.join(
    results_dir,
    "training_loss.png"
)

plt.savefig(
    loss_plot_path,
    dpi=300
)

plt.show()


# ============================================================
# 12. BASIC TEST EVALUATION
# ============================================================

print("\n" + "=" * 70)
print("TESTING TRAINED MODEL")
print("=" * 70)

test_loss, test_accuracy = model.evaluate(
    X_test_cnn,
    y_test_cat,
    verbose=1
)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")


# ============================================================
# 13. SAVE FINAL TRAINING SUMMARY
# ============================================================

summary_path = os.path.join(
    results_dir,
    "training_summary.txt"
)

with open(summary_path, "w") as file:

    file.write(
        "AI Based Intelligent RF Spectrum Signal Identification System\n"
    )

    file.write(
        "CNN Model Training Summary\n"
    )

    file.write("=" * 60 + "\n\n")

    file.write(
        f"Number of Epochs Completed: {len(history.history['accuracy'])}\n"
    )

    file.write(
        f"Final Training Accuracy: "
        f"{history.history['accuracy'][-1] * 100:.2f}%\n"
    )

    file.write(
        f"Final Validation Accuracy: "
        f"{history.history['val_accuracy'][-1] * 100:.2f}%\n"
    )

    file.write(
        f"Test Accuracy: "
        f"{test_accuracy * 100:.2f}%\n"
    )

    file.write(
        f"Test Loss: {test_loss:.4f}\n"
    )

print("\nTraining summary saved:")
print(os.path.abspath(summary_path))


# ============================================================
# 14. FINAL RESULT
# ============================================================

print("\n" + "=" * 70)
print("MODEL TRAINING COMPLETED SUCCESSFULLY")
print("=" * 70)

print(
    f"\nEpochs Completed: "
    f"{len(history.history['accuracy'])}"
)

print(
    f"Final Training Accuracy: "
    f"{history.history['accuracy'][-1] * 100:.2f}%"
)

print(
    f"Final Validation Accuracy: "
    f"{history.history['val_accuracy'][-1] * 100:.2f}%"
)

print(
    f"Test Accuracy: "
    f"{test_accuracy * 100:.2f}%"
)

print("\nSaved Files:")

print("- Best Model:")
print(f"  {best_model_path}")

print("- Final Model:")
print(f"  {final_model_path}")

print("- Training History:")
print(f"  {history_path}")

print("- Accuracy Graph:")
print(f"  {accuracy_plot_path}")

print("- Loss Graph:")
print(f"  {loss_plot_path}")

print("- Training Summary:")
print(f"  {summary_path}")

print("\nNEXT STAGE: DETAILED MODEL EVALUATION")